# Reformat Emmer Data

In [15]:
import json, urllib.request
from pathlib import Path
import math

PROJECT_ROOT = Path("../").resolve()
OUT_DIR = PROJECT_ROOT / "tracks" / "new_data"
INPUT_DIR = PROJECT_ROOT / "tracks" / "raw_emmer"

GENOME = "hg38"
chrom, start1, end1 = "chr19", 11_039_463, 11_163_820   # 1-based inclusive, like the browser

# read excel files
import pandas as pd
import openpyxl
import csv

In [16]:
# 1. Map sheet names to your desired output TSV filenames
sheet_mapping = {
    "ATAC_LFC": f"{OUT_DIR}/ATAC_liver_v2_mean_log2FC.bedGraph",
    "ATAC_peaks": f"{OUT_DIR}/ATAC_liver_v2_peaks.bed",
    "DHS_signal": f"{OUT_DIR}/dhs_liver_v2_mean_log2FC.bedGraph",
    "DHS_peaks": f"{OUT_DIR}/dhs_liver_v2_peaks.bed",
    "H3K27ac_LFC": f"{OUT_DIR}/H3K27ac_liver_v2_mean_log2FC.bedGraph",
    "H3K27ac_peaks": f"{OUT_DIR}/H3K27ac_liver_v2_peaks.bed",
    "H3K4me1_LFC": f"{OUT_DIR}/H3K4me1_liver_v2_mean_log2FC.bedGraph",
    "H3K4me1_peaks": f"{OUT_DIR}/H3K4me1_liver_v2_peaks.bed",
}

exponentiate = lambda x: 2 ** x if x is not None else None
exponentiate_sheets_2_row = {
    #"ATAC_LFC":1, 
    "DHS_signal":1, 
    #"H3K27ac_LFC":1, 
    #"H3K4me1_LFC":1
    }

# 2. Load the workbook (data_only=True evaluates formulas to values)
wb = openpyxl.load_workbook(f"{INPUT_DIR}/TableS5_CRE-predictors.xlsx", data_only=True)

# 3. Process specified sheets
for sheet_name, output_filename in sheet_mapping.items():
    if sheet_name not in wb.sheetnames:
        print(f"Warning: '{sheet_name}' not found in workbook. Skipping.")
        continue

    ws = wb[sheet_name]

    with open(
        output_filename, mode="w", newline="", encoding="utf-8"
    ) as tsv_file:
        writer = csv.writer(tsv_file, delimiter="\t")

        for i,row in enumerate(ws.iter_rows(values_only=True)):
            # Skip completely empty rows
            if not any(cell is not None for cell in row):
                continue

            # Apply exponentiation to the 4th column (index 3) if the row is equal to or greater than the specified row for that sheet
            if sheet_name in exponentiate_sheets_2_row and i >= exponentiate_sheets_2_row[sheet_name]:
                row = list(row)  # Convert tuple to list for mutability
                row[3] = exponentiate(row[3])  # Exponentiate the 4th column

            # Replace None with empty string
            clean_row = ["" if cell is None else str(cell) for cell in row]
            writer.writerow(clean_row)

    print(f"Saved '{sheet_name}' -> '{output_filename}'")

Saved 'ATAC_LFC' -> 'C:\Users\craut\Documents\MM_LDLR_func_viewer\tracks\new_data/ATAC_liver_v2_mean_log2FC.bedGraph'
Saved 'ATAC_peaks' -> 'C:\Users\craut\Documents\MM_LDLR_func_viewer\tracks\new_data/ATAC_liver_v2_peaks.bed'
Saved 'DHS_signal' -> 'C:\Users\craut\Documents\MM_LDLR_func_viewer\tracks\new_data/dhs_liver_v2_mean_log2FC.bedGraph'
Saved 'DHS_peaks' -> 'C:\Users\craut\Documents\MM_LDLR_func_viewer\tracks\new_data/dhs_liver_v2_peaks.bed'
Saved 'H3K27ac_LFC' -> 'C:\Users\craut\Documents\MM_LDLR_func_viewer\tracks\new_data/H3K27ac_liver_v2_mean_log2FC.bedGraph'
Saved 'H3K27ac_peaks' -> 'C:\Users\craut\Documents\MM_LDLR_func_viewer\tracks\new_data/H3K27ac_liver_v2_peaks.bed'
Saved 'H3K4me1_LFC' -> 'C:\Users\craut\Documents\MM_LDLR_func_viewer\tracks\new_data/H3K4me1_liver_v2_mean_log2FC.bedGraph'
Saved 'H3K4me1_peaks' -> 'C:\Users\craut\Documents\MM_LDLR_func_viewer\tracks\new_data/H3K4me1_liver_v2_peaks.bed'
